# Development corpus EDA

Start with the read-only pipeline dry-run below. Continue to EDA or copy the printed real command only when the decision allows execution. This notebook never launches training automatically and does not display raw comments.

## Mandatory preflight (dry-run)

Set the pipeline configuration path, then run the next two cells. `BLOCKED` means the real pipeline must not start; `WARN` requires conscious review. The check writes no files and starts no subprocesses.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.operations import PipelineConfig, dry_run_pipeline, render_dry_run_report  # noqa: E402

PIPELINE_CONFIG_PATH = PROJECT_ROOT / "configs/pipeline.example.json"
pipeline_config = PipelineConfig.model_validate_json(
    PIPELINE_CONFIG_PATH.read_text(encoding="utf-8")
)

In [ ]:
dry_run_report = dry_run_pipeline(
    pipeline_config,
    PROJECT_ROOT,
    config_path=PIPELINE_CONFIG_PATH,
)
rendered_dry_run_report = render_dry_run_report(dry_run_report)
if not dry_run_report.can_run:
    message = "Dry-run is BLOCKED; resolve reported checks before a real run"
    raise RuntimeError(message)
rendered_dry_run_report

The final `Command:` line is the exact command for a deliberate terminal launch. It is shown for review only: the notebook does not execute it. Re-run preflight after changing data, configuration, resume state, or restart options.

In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROFILE_PATH = PROJECT_ROOT / "data/reports/eda/development-profile.json"
if not PROFILE_PATH.is_file():
    message = "Run scripts/run_eda.py first"
    raise FileNotFoundError(message)
profile = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))

## Corpus overview

In [ ]:
{key: profile[key] for key in [
    "records", "unique_texts", "duplicate_texts", "duplicate_groups",
    "unique_authors", "unique_videos", "unique_channels", "unique_queries",
]}

## Length, language, noise, and time aggregates

In [ ]:
{key: profile[key] for key in [
    "character_lengths", "token_lengths", "languages",
    "noise_categories", "monthly_records",
]}

## Interpretation checklist

- Review p50/p95/p99 lengths before selecting cleaning and embedding limits.
- Treat noise categories as overlapping diagnostics, not additive removal counts.
- Validate language labels manually on the local development-only sample before filtering.
- Keep validation and test unopened until their planned evaluation stages.
- Do not save cells containing raw comments into this tracked notebook.

## Stage 5 embedding artifacts

Load only the manifest and memory-mapped matrix here. Keep raw comment values out of the tracked notebook.

In [ ]:
import numpy as np

EMBEDDING_DIR = PROJECT_ROOT / "data/processed/embeddings"
embedding_manifest = json.loads((EMBEDDING_DIR / "embedding-manifest.json").read_text())
embeddings = np.load(EMBEDDING_DIR / "embeddings.npy", mmap_mode="r", allow_pickle=False)
{"shape": embeddings.shape, "dtype": str(embeddings.dtype), "model": embedding_manifest["model_name"]}

## Stage 7 final corpus

Stage 8 reads the deduplicated matrix through memory mapping and verifies its shape against the manifest.

In [ ]:
CORPUS_DIR = PROJECT_ROOT / "data/processed/corpus"
corpus_manifest = json.loads((CORPUS_DIR / "corpus-manifest.json").read_text())
final_embeddings = np.load(CORPUS_DIR / "final-embeddings.npy", mmap_mode="r", allow_pickle=False)
expected_shape = (corpus_manifest["stats"]["output_records"], corpus_manifest["dimensions"])
{
    "shape": final_embeddings.shape,
    "matches_manifest": final_embeddings.shape == expected_shape,
    "removed": corpus_manifest["stats"]["removed_semantic_duplicates"],
}

## Stage 8 UMAP visualization

The 2D space is diagnostic only. This plot uses coordinates and row indexes, never raw comment text.

In [ ]:
import matplotlib.pyplot as plt

UMAP_DIR = PROJECT_ROOT / "data/processed/umap"
visualization_manifest = json.loads((UMAP_DIR / "visualization-manifest.json").read_text())
coordinates = np.load(UMAP_DIR / "visualization-2d.npy", mmap_mode="r", allow_pickle=False)
plot_limit = min(len(coordinates), 50_000)
figure, axis = plt.subplots(figsize=(10, 7))
axis.scatter(coordinates[:plot_limit, 0], coordinates[:plot_limit, 1], s=2, alpha=0.25)
axis.set(title="UMAP diagnostic projection", xlabel="UMAP-1", ylabel="UMAP-2")
figure

## Stage 9 HDBSCAN diagnostics

Labels color the separate 2D diagnostic projection. No comment text is loaded or attached to points.

In [ ]:
CLUSTERING_DIR = PROJECT_ROOT / "data/processed/clustering"
clustering_manifest = json.loads((CLUSTERING_DIR / "clustering-manifest.json").read_text())
labels = np.load(CLUSTERING_DIR / "cluster-labels.npy", mmap_mode="r", allow_pickle=False)
probabilities = np.load(CLUSTERING_DIR / "cluster-probabilities.npy", mmap_mode="r", allow_pickle=False)
diagnostic_rows = min(len(labels), len(coordinates), 50_000)
figure, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].scatter(
    coordinates[:diagnostic_rows, 0],
    coordinates[:diagnostic_rows, 1],
    c=labels[:diagnostic_rows],
    s=2,
    alpha=0.3,
)
cluster_sizes = np.bincount(labels[labels >= 0])
axes[1].hist(cluster_sizes, bins=min(50, max(1, len(cluster_sizes))))
axes[0].set_title("HDBSCAN labels on diagnostic UMAP")
axes[1].set(title="Cluster-size distribution", xlabel="Records", ylabel="Clusters")
{
    "clusters": clustering_manifest["metrics"]["clusters"],
    "outlier_share": clustering_manifest["metrics"]["outlier_share"],
    "mean_probability": float(np.mean(probabilities)),
}

## Stage 10 topic representation diagnostics

Load aggregate representations only. Representative records are referenced by row index; do not save raw text in this notebook.

In [ ]:
TOPICS_DIR = PROJECT_ROOT / "data/processed/topics"
topic_manifest = json.loads((TOPICS_DIR / "topic-representation-manifest.json").read_text())
topic_rows = [
    json.loads(line)
    for line in (TOPICS_DIR / "topic-representations.jsonl").read_text().splitlines()
    if line.strip()
]
topic_table = [
    {
        "topic_id": row["topic_id"],
        "name": row["name"],
        "records": row["records"],
        "mean_probability": row["mean_probability"],
    }
    for row in topic_rows
]
sorted(topic_table, key=lambda row: -row["records"])[:20]

## Stage 11 outlier reassignment diagnostics

Inspect aggregate decisions and indexes only. Keep raw comment text out of this tracked notebook.

In [ ]:
REASSIGNMENT_DIR = PROJECT_ROOT / "data/processed/outlier-reassignment"
reassignment_manifest = json.loads(
    (REASSIGNMENT_DIR / "outlier-reassignment-manifest.json").read_text()
)
final_labels = np.load(
    REASSIGNMENT_DIR / "final-cluster-labels.npy", mmap_mode="r", allow_pickle=False
)
reassignment_metrics = reassignment_manifest["metrics"]
figure, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(
    ["reassigned", "remaining"],
    [reassignment_metrics["reassigned_outliers"], reassignment_metrics["remaining_outliers"]],
)
diagnostic_rows = min(len(final_labels), len(coordinates), 50_000)
axes[1].scatter(
    coordinates[:diagnostic_rows, 0],
    coordinates[:diagnostic_rows, 1],
    c=final_labels[:diagnostic_rows],
    s=2,
    alpha=0.3,
)
axes[0].set_title("Outlier decisions")
axes[1].set_title("Final labels on diagnostic UMAP")
reassignment_metrics